In [ ]:
# Cell 1: clone repo va sync dependency bang uv
import os
from pathlib import Path

%cd /content

REPO_URL = "https://github.com/vietanh-io/ai-video-content-management-system.git"
BRANCH_NAME = "ai-service/chaptering"  # doi thanh branch da push neu can
REPO_DIR = Path("/content/vid-pilot")

if not Path("/root/.local/bin/uv").exists():
    !curl -LsSf https://astral.sh/uv/install.sh | sh

os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]

if not REPO_DIR.exists():
    !git clone --branch {BRANCH_NAME} {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch origin {BRANCH_NAME}
    !git checkout {BRANCH_NAME}
    !git pull --ff-only origin {BRANCH_NAME}

%cd /content/vid-pilot/ai-service
!uv sync


In [ ]:
# Cell 2: config. HF token lay tu Colab Secrets.
import os
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN") or userdata.get("HUGGING_FACE_HUB_TOKEN") or ""

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
os.environ["PYANNOTE_AUTH_TOKEN"] = HF_TOKEN

os.environ["ASR_PROVIDER"] = "faster-whisper"
os.environ["ASR_MODEL_SIZE"] = "small"
os.environ["ASR_LANGUAGE"] = "vi"
os.environ["ASR_DEVICE"] = "cpu"
os.environ["ASR_COMPUTE_TYPE"] = "int8"

os.environ["AUDIO_DECODER_PROVIDER"] = "torchaudio"
os.environ["VAD_PROVIDER"] = "silero"
os.environ["DIARIZATION_PROVIDER"] = "pyannote"
os.environ["DIARIZATION_DEVICE"] = "cpu"
os.environ["SOURCE_SEPARATION_PROVIDER"] = "noop"

print("Config xong. HF token:", "co" if HF_TOKEN else "chua co")


In [ ]:
# Cell 3: chon file trong Google Drive: MyDrive/transcript_smoke/transcripts/audio/filename
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

BASE_DIR = Path("/content/drive/MyDrive/transcript_smoke/transcripts")
AUDIO_DIR = BASE_DIR / "audio"
RESULT_DIR = BASE_DIR / "result"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

FILENAME = "your_audio_file.wav"  # sua ten file o day
AUDIO_PATH = AUDIO_DIR / FILENAME
RESULT_PATH = RESULT_DIR / f"{Path(FILENAME).stem}.json"

assert AUDIO_PATH.exists(), f"Khong thay file: {AUDIO_PATH}"
print("Audio:", AUDIO_PATH)
print("Result:", RESULT_PATH)


In [ ]:
# Cell 4: goi pipeline va ghi ket qua ra transcript_smoke/transcripts/result/filename
import json
import os
import subprocess
from pathlib import Path

RUNNER = Path("/content/vid-pilot/ai-service/transcript_smoke_runner.py")
RUNNER.write_text(
    r'''
import json
import os
import sys
from pathlib import Path

from app.core.config import get_settings
from app.runtime.container import build_transcription_workflow
from app.schemas.transcription_request import TranscriptionOptionsInput, TranscriptionRequest

audio_path = Path(sys.argv[1])
result_path = Path(sys.argv[2])
language = os.environ.get("TRANSCRIPT_LANGUAGE", os.environ.get("ASR_LANGUAGE", "vi"))
pipeline = os.environ.get("TRANSCRIPT_PIPELINE", "vad-chunked")
enable_word_timestamps = os.environ.get("TRANSCRIPT_ENABLE_WORD_TIMESTAMPS", "true").lower() == "true"

enable_diarization = pipeline == "diarized-turns"

workflow = build_transcription_workflow(get_settings())
result = workflow.execute(
    TranscriptionRequest(
        request_id=f"colab-{audio_path.stem}",
        local_path=audio_path,
        filename=audio_path.name,
        content_type="audio/*",
        options=TranscriptionOptionsInput(
            language=language,
            enable_vad=True,
            enable_diarization=enable_diarization,
            enable_source_separation=False,
            enable_word_timestamps=enable_word_timestamps,
        ),
    )
)

result_path.parent.mkdir(parents=True, exist_ok=True)
data = result.model_dump()
result_path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

print("Saved:", result_path)
print("Language:", data["language"])
print("ASR model:", data["asr_model"])
word_count = sum(len(segment.get("words", [])) for segment in data["segments"])
print("Segments:", len(data["segments"]))
print("Words:", word_count)
print("Word timestamps:", "enabled" if enable_word_timestamps else "disabled")
print("\nFull text:\n")
print(data["full_text"])
''',
    encoding="utf-8",
)

env = os.environ.copy()
env["TRANSCRIPT_PIPELINE"] = "vad-chunked"  # doi thanh "diarized-turns" neu muon test pipeline diarization
env["TRANSCRIPT_LANGUAGE"] = "en"
env["TRANSCRIPT_ENABLE_WORD_TIMESTAMPS"] = "true"

completed = subprocess.run(
    ["uv", "run", "python", str(RUNNER), str(AUDIO_PATH), str(RESULT_PATH)],
    cwd="/content/vid-pilot/ai-service",
    env=env,
    text=True,
    capture_output=True,
)

print(completed.stdout)
if completed.stderr:
    print("STDERR:")
    print(completed.stderr)

if completed.returncode != 0:
    raise RuntimeError(f"Transcript pipeline failed with exit code {completed.returncode}")
